# 09c · 全蛋白组打分

## 这一章在整个项目里的位置

```
00–08 ✅ → [09 上线] → 10 对外 API
              │
              ├─ 09a 定版和考试 ✅
              ├─ 09b 注册模型 ✅
              └─ 09c 全蛋白组打分：每一个可能突变的两个分数   ← 你在这里
```

## 背景：为什么需要这一章

**这就是整个项目的最终目标**：重新预测全蛋白组。人类参考蛋白组有 20,652 个蛋白、1,146 万个残基，每个残基可以换成另外 19 种氨基酸，一共约 **2.18 亿个**可能的错义突变。这一章给它们每一个打上两个分数：

| 分数 | 回答的问题 | 来自 |
|---|---|---|
| `stability_score` | 会不会让蛋白不稳定 | 09b 注册的稳定性网络 |
| `function_score` | 会不会让蛋白失去功能 | 零样本 + 树模型 + 稳定性网络的集成 |

两个都在 0 到 1 之间，**越低越可能有害**。打完分，用一个模型完全没见过的标准检验一下：**ClinVar** 里医生判过致病或良性的突变。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2 步：Databricks 建全蛋白组的残基特征，下载到你的 Mac（半小时到一小时，看网速）
   ├─ 第 5 步：在你的 Mac 后台给全部蛋白打分（约 5–6 小时；你用电脑时它会自动让路）
   ├─ 第 6 步：结果传回平台，变成产品表（约 15–30 分钟）
   └─ 第 7、8 步：ClinVar 外部验证，新旧预测对比
看结果：这里的输出；Databricks 网页的 Catalog → gold → fct_variant_effect
```

## 这一章做的 8 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 全蛋白组要算什么、在哪算 | 生产线的布局 |
| 2 | 全蛋白组的残基特征 | 原料备齐 |
| 3 | 训练和上线的特征必须一样 | 生产配方和实验配方逐项核对 |
| 4 | 先试 3 个蛋白 | 小批量试产 |
| 5 | 后台跑全部 | 正式开工 |
| 6 | 结果变成产品表 | 入库上架 |
| 7 | ClinVar 外部验证 | 第三方检测 |
| 8 | 新旧对比 | 和上一代产品比 |

**花费：0**。**时间**：一个晚上。大部分在第 5 步，可以开着电脑过夜。

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # PyTorch 和 XGBoost 在同一个进程里（见 07f）
import subprocess, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from transformers.utils import logging as hf_logging
from qafi_platform import lakehouse, registry, scoring
from qafi_platform.lakehouse import sql
from qafi_core import proteome, release
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
MODELS = ROOT / "data/models"
OUT = ROOT / "data/proteome"                      # 打分结果放这里，不进 Git
RELEASE = time.strftime("%Y-%m-%d")                # 这一版产品的名字
AMINO, BLOSUM = pd.read_csv(ROOT / "dbt/seeds/amino_acid.csv"), pd.read_csv(ROOT / "dbt/seeds/blosum62.csv")
print("Databricks:", w.config.host, "| 这一版:", RELEASE)

---

## 第 1 步 · 全蛋白组要算什么、在哪算

### 在干什么
弄清楚 2.18 亿个分数是怎么算出来的，每一步在哪台机器上算。

### 背景
比喻：**生产线的布局**。一个突变的两个分数要经过五道工序：

```
                      [Databricks]                                   [你的 Mac]
gold.dim_residue_proteome ──下载──►  1. ESM-2 读每个蛋白，得到每个残基的向量（要 GPU）
（每个残基的特征，同一段 SQL）       2. 稳定性网络：向量 + 10 个残基特征 → 20 种替换各一个分
                                     3. 算出 19 种替换的 16 个特征 → 树模型打分
                                     4. 三个成员换成百分位 → 两个分数
                                     5. 每 50 个蛋白存一份 parquet
                                                │
gold.fct_variant_effect ◄──装表── ml.proteome_scores ◄──上传──┘
（产品：分数 + 基因名）
```

**为什么第 1–5 道在 Mac 上**：第 1 道要 GPU，Databricks 免费版没有（07e 发现的）。你的 Mac 有。第 2–4 道很轻，顺手一起做，省得来回传数据。

**为什么不把 ESM 向量存下来**：全蛋白组的向量是 1,146 万 × 1,280 ≈ 147 亿个数，存下来要**几十 GB**。不存，边算边用，用完就扔，只留下分数。这叫**流式推理**。

### 用了什么技术，为什么
- 模型从**注册表**按 `@champion` 取（09b），换算表跟着模型走
- 打分程序可以**断点续跑**：每 50 个蛋白存一份，先存分数、再存一份"这 50 个做完了"的清单。中途停了（关机、内存紧张），下次从没做完的地方接着做
- **对别的程序友好**：用最低的优先级跑（你的程序先用处理器）；macOS 的内存压力到了"危急"，就把模型卸掉、等 5 分钟再试。和 07e 的回填是同一套办法，只有一点不同：07e 要等压力"正常"才开始，而开着 VS Code 的 Mac 常常一直是"警告"，所以这里只要不是"危急"就开始
- **GPU 上的形状固定**：每个窗口补齐到几个固定长度之一，每批都凑满（08b 的教训）

### 面试怎么说
> "Proteome scoring is streaming inference on a laptop GPU: ESM-2 embeddings are computed per chunk of proteins and consumed immediately by the released models, so tens of gigabytes of embeddings are never stored. The job is resumable, yields to other programs under memory pressure, and keeps GPU shapes fixed."

---

## 第 2 步 · 全蛋白组的残基特征

### 在干什么
在 Databricks 上给全部 20,652 个蛋白算残基特征，下载到 Mac。

### 背景
比喻：**原料备齐**。稳定性网络要每个残基的 10 个特征（pLDDT、rsa、熵……），树模型还要 ESM 的 20 个概率。训练时这些来自 `gold.dim_residue`，只有 494 个有标签的蛋白。

**同一段 SQL**：我把 `dim_residue` 的 SQL 抽成了一个 dbt **宏**（macro，可以复用的 SQL 片段）`residue_features`。现在有两张表共用它：

| 表 | 蛋白 | 用途 |
|---|---|---|
| `gold.dim_residue` | 494 个有标签的 | 训练（07f 起） |
| `gold.dim_residue_proteome` | 全部 20,652 个 | 上线打分（这一章） |

两张表的每一个数都是同一段代码算的。07a 说的 training-serving skew，在这一层就堵住了。

### 执行
建表（约 2 分钟），再下载 1,146 万行（约 20–50 分钟，看网速；8 路并行下载，网络断了会自己重试）：

In [ ]:
os.environ.update(lakehouse.dbt_env(w))
run("cd dbt && uv run dbt build --select dim_residue_proteome --profiles-dir .")
columns = ", ".join(["uniprot_id", "position", "aa", "logprobs", *release.RESIDUE_FEATURES])
rows = lakehouse.download(w, f"SELECT {columns} FROM {CATALOG}.gold.dim_residue_proteome ORDER BY uniprot_id, position",
                          MODELS / "proteome_residues.parquet")
lakehouse.download(w, f"SELECT uniprot_id, sequence FROM {CATALOG}.gold.dim_protein", MODELS / "proteome_sequences.parquet")
print(f"{rows:,} 个残基，{(MODELS / 'proteome_residues.parquet').stat().st_size / 1e9:.1f} GB")

### 结果怎么看
- dbt 最后一行 `PASS=5`：表建好了，"一个残基一行"等检查都过了
- **11,456,418** 个残基，约 1.4 GB

---

## 第 3 步 · 训练和上线的特征必须一样

### 在干什么
确认 Mac 上算出来的 16 个突变特征，和训练时 SQL 算的一模一样。

### 背景
比喻：**生产配方和实验配方逐项核对**。残基特征两边都来自同一张 SQL 表，没问题。但**突变特征**（BLOSUM62、体积变化、ESM 打分差……）在训练时是 SQL 算的（`fct_variant_features`），在 Mac 上要用 Python 重新算一遍，因为 2.18 亿行的表在免费版上建不起。

**同一个特征有两份代码**，这正是 training-serving skew 最常见的来源：比如 SQL 里体积差是"新减旧"，Python 里不小心写成"旧减新"，模型照样出分，只是全错了，而且没有任何报错。

办法：拿 84 万个训练突变，两边各算一遍，**逐个比对**。

### 执行

In [ ]:
lakehouse.download(w, f"""SELECT {columns} FROM {CATALOG}.gold.dim_residue WHERE uniprot_id IN
                          (SELECT uniprot_id FROM {CATALOG}.gold.protein_split WHERE split = 'train')""", MODELS / "train_residues_features.parquet")
lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'train'", MODELS / "train_variants.parquet")
difference = proteome.check_features(pd.read_parquet(MODELS / "train_residues_features.parquet"),
                                     pd.read_parquet(MODELS / "train_variants.parquet"), AMINO, BLOSUM)
assert (difference < 1e-4).all(), difference
difference.rename("Python 和 SQL 的最大差别").to_frame()

### 结果怎么看
16 个特征，每个一行，是两边在 84 万个突变上的**最大**差别：
- 大多数是 **0**，一位不差
- 体积相关的几个是 0.00003 上下：SQL 用 64 位小数，Python 用 32 位，这是舍入，不是算法不同
- 上面那句 `assert` 保证：哪天有人改了一边没改另一边，这一格会直接报错

### 公司里什么情况怎么做
- 能用同一段代码就用同一段（我们的残基特征就是）；做不到时，**用测试证明两段代码算出来一样**。这种测试叫**一致性测试**（parity test）
- 这个测试要放在每次上线之前跑，而不是只跑一次

### 面试怎么说
> "Residue features come from one SQL macro for both training and serving. Variant features had to be recomputed in Python for the proteome, so a parity check compares them with the SQL feature table on all 841,000 training variants before scoring; the largest difference is float32 rounding."

---

## 第 4 步 · 先试 3 个蛋白

### 在干什么
小批量试产：用注册表里的冠军模型，给 3 个熟悉的蛋白打分，看看结果是否合理、速度多少。

### 背景
挑了三个：
- **PTEN**（训练蛋白）：我们能拿 09a 的算法再算一遍，核对新程序给出同样的分
- **TPMT**（训练蛋白）
- **TP53**（考试蛋白）：几个著名的癌症热点突变，看看分数。**只看分数，不看它的标签**

### 执行

In [ ]:
lm = proteome._language_model()
esm, tokenizer = lm.load()
MODELS_IN_USE = {"esm": esm, "tokenizer": tokenizer,
                 "head": registry.load(registry.full_name("stability"), flavor="pytorch").to(next(esm.parameters()).device),
                 "trees": registry.load(registry.full_name("trees"), flavor="xgboost"),
                 "tables": registry.distributions(registry.full_name("trees"))}
sequences = pd.read_parquet(MODELS / "proteome_sequences.parquet").set_index("uniprot_id")["sequence"]
trial = {p: sequences[p] for p in ["P60484", "P51580", "P04637"]}
residues = pd.read_parquet(MODELS / "proteome_residues.parquet", filters=[("uniprot_id", "in", list(trial))])
started = time.time()
tried = proteome.score_proteins(trial, residues, MODELS_IN_USE, AMINO, BLOSUM)
seconds = time.time() - started
print(f"{len(tried):,} 个突变（{sum(map(len, trial.values())):,} 个残基 × 19），{seconds:.0f} 秒")
tp53 = tried[tried.uniprot_id == "P04637"].set_index(["position", "alt_aa"])
pd.DataFrame({m: tp53.loc[(int(m[1:-1]), m[-1]), ["stability_score", "function_score"]]
              for m in ["R175H", "G245S", "R248Q", "R273H", "P72R"]}).T.round(2)

### 结果怎么看
**数量对得上**：1,041 个残基 × 19 = 19,779 个突变，一个不少。

**TP53 的五个突变**（两个分数都是越低越可能有害；下面是我写这一章时的数字，你在 09a 重新调参训练过，会略有不同）：

| 突变 | 是什么 | 稳定性分 | 功能分 |
|---|---|---|---|
| R175H | 结构性热点，突变让 DNA 结合域折叠不好 | 约 0.22 | 约 0.30 |
| G245S | 结构性热点 | 约 0.36 | 约 0.32 |
| R248Q | **DNA 接触**热点：直接碰 DNA 的那个精氨酸 | 约 0.90 | 约 0.67 |
| R273H | **DNA 接触**热点 | 约 0.77 | 约 0.47 |
| P72R | 人群中常见的良性多态性 | 约 0.69 | 约 0.72 |

**这张表很能说明我们模型懂什么、不懂什么：**
- 结构性热点（R175H、G245S）：两个分数都低，判对了
- DNA 接触热点（R248Q、R273H）：蛋白本身折叠得好好的（稳定性分高），坏在**不能结合 DNA 了**。我们的模型主要从稳定性数据里学，**看不出这种功能位点**。功能分只是中等
- 良性的 P72R：两个分数都高，判对了

这正是 08 章反复出现的结论：**稳定性学会了，功能还差一截**。用户拿到我们的分数，要知道这一点。

### 公司里什么情况怎么做
- 上线前拿几个**大家都知道答案**的例子看一眼，比任何汇总指标都更快发现问题
- 发现模型的盲区，要写进说明文档，而不是藏起来

### 面试怎么说
> "Before the full run I scored three proteins: counts matched exactly, a training protein reproduced the scores of the evaluation code, and TP53 showed the model's blind spot: structural hotspots like R175H score as damaging, but DNA-contact hotspots like R248Q do not, because the training data is mostly about stability."

---

## 第 5 步 · 后台跑全部

### 在干什么
在你的 Mac 后台，给 20,652 个蛋白全部打分。

### 背景
我写这一章时用 300 个蛋白测过速度：每秒约 **580** 个残基，全部 1,146 万个残基，**约 5–6 小时**（电脑空着时；你用电脑时它会让路，会更久）。

所以这一步不在 notebook 里等，而是**在后台启动一个独立的程序**（`python -m qafi_platform.scoring`），和 07e 的回填一样：
- 关掉这个 notebook、甚至合上电脑盖子再打开，都不影响它（合盖会暂停，打开接着跑）
- 它每做完 50 个蛋白就存一份。万一中途停了，**再运行一次下面这一格**，它会从没做完的地方接着做
- 它的进度写在 `data/proteome/run.log` 里

### 执行
先把刚才试跑占的内存还回去，再在后台启动：

In [ ]:
import gc
del esm, MODELS_IN_USE
gc.collect(); lm.free_memory()
OUT.mkdir(parents=True, exist_ok=True)
subprocess.run(f"nohup caffeinate -i nice -n 19 uv run python -m qafi_platform.scoring {MODELS / 'proteome_residues.parquet'} "
               f"{MODELS / 'proteome_sequences.parquet'} {OUT} >> {OUT / 'run.log'} 2>&1 &", shell=True)
print("已在后台启动；进度看下一格")

`caffeinate -i` 让 Mac 在它跑的时候不进入睡眠；`nice -n 19` 让它把处理器优先让给你的其他程序。

随时运行下面这一格看进度（最后几行日志，和已经做完的蛋白数）：

In [ ]:
print(subprocess.run(["tail", "-3", OUT / "run.log"], capture_output=True, text=True).stdout)
print(f"已完成 {len(proteome.done(OUT)):,} / {len(sequences):,} 个蛋白")

### 结果怎么看
日志的每一行：`做了多少个 of 一共多少个, 每秒多少个残基, 还要多少小时`。

- 如果第一行是 `waiting until memory pressure is below critical`：内存太紧，它在等。关掉别的占内存的程序（其他 notebook 的内核、Docker、很多浏览器标签），几分钟内就会开始
- 如果看到 `memory is critical: model unloaded`，说明你那边的程序占了太多内存，它主动让路了，过 5 分钟会自己再试
- 最后一行出现 `all proteins scored` 就是全部做完了。**做完以后再往下运行**

### 公司里什么情况怎么做
- 长任务一定要**能断点续跑**，并且**进度可见**。跑了 6 小时在第 5 小时崩了、只能从头来，是最常见的浪费
- 在公司里，这一步会交给集群上的一个批处理任务（GPU 节点），而不是一台笔记本。程序的写法是一样的：分块、可续跑、结果写成 parquet

---

## 第 6 步 · 结果变成产品表

### 在干什么
把 Mac 上的分数传到平台，装成表，再加一个带基因名的产品视图。

### 背景
三样东西：
1. **上传**：几百个 parquet 文件，传到 `ml.files` 存储区（约 3.4 GB）。一个文件一个文件传：网络断了会自己重试；整格失败了，再运行一次，已经传好的文件会跳过
2. **装表**：`ml.proteome_scores`，一个突变一行，每行还记着是哪一版、用的哪个模型版本。按蛋白聚簇，查一个基因很快
3. **产品视图**：`gold.fct_variant_effect`，在分数旁边加上基因名、`R175H` 这样的写法。10 号的 API 直接查它。视图不占存储，查的时候才算

### 执行

In [ ]:
assert len(proteome.done(OUT)) == len(sequences), "还没做完：等第 5 步的日志出现 all proteins scored"
target = scoring.upload(w, OUT, RELEASE)              # 断了就再运行这一格：传过的文件会跳过
print("已上传到", target, "| 用的模型版本:", registry.champions())
display(scoring.load(w, RELEASE, registry.champions()))
run("cd dbt && uv run dbt build --select fct_variant_effect --profiles-dir .")
sql(w, f"""SELECT gene_name, variant, round(stability_score, 2) AS stability, round(function_score, 2) AS function
           FROM {CATALOG}.gold.fct_variant_effect WHERE gene_name = 'TP53' AND variant IN ('R175H', 'R248Q', 'P72R')""")

### 结果怎么看
- 第一张表：`variants` 约 **2.18 亿**，`proteins` **20,652**，`without_function_score` 应该是 0
- dbt 建好了产品视图
- 最后一张表：从产品视图里查 TP53 的三个突变，和第 4 步试跑的分数一样

### 去网页上看
Databricks → **Catalog** → `qafi_dev` → `gold` → `fct_variant_effect`：**Sample Data** 里能翻到任何一个基因的突变；**Lineage** 一路追到 `ml.proteome_scores`、注册表里的模型、再到训练数据。

### 面试怎么说
> "The scores are loaded as a clustered Delta table with the release and model versions on every row, and exposed through a dbt view that adds gene names for the API."

---

## 第 7 步 · ClinVar 外部验证

### 在干什么
用医生的诊断检验我们的分数：ClinVar 里判为致病和良性的突变，我们的分数能不能把它们分开。

### 背景
比喻：**第三方检测**。前面所有的成绩都来自 DMS 实验。ClinVar 是另一种完全独立的证据：临床上见到的突变，由医生和遗传学家判定致病还是良性。

我从平台上的 ClinVar 原始文件（04 号装进来的）整理出了 `gold.fct_clinvar_missense`：
- 胚系、单核苷酸、至少一颗星的审核；致病 / 可能致病 vs 良性 / 可能良性
- 从名字里读出蛋白上的变化（比如 `p.Arg175His`），并且核对 UniProt 序列上那个位置确实是 R
- **只用我们的数据里完全没有的蛋白**：训练蛋白和考试蛋白都排除。这样 ClinVar 上的成绩一点都没有"见过"的成分

一共约 **16.8 万**个突变（4.9 万致病、11.9 万良性），分布在一万多个蛋白上。

**指标是 AUROC**：随便抽一个致病突变和一个良性突变，致病那个分数更低的概率。0.5 = 瞎猜，1 = 完美。

**两种比法**：
- `own coverage`：每个预测器在它自己有分的所有突变上
- `shared variants`：只在**所有预测器都有分**的突变上，这才是公平的同题比较

别人的分数（AlphaMissense、EVE、popEVE、CPT-1、旧 QAFI2）来自 09a 第 2 步装好的 `silver.variant_scores`。

### 执行

In [ ]:
clinvar = scoring.clinvar(w, MODELS / "clinvar_scores.parquet")
table = scoring.auroc(clinvar)
print(f"{len(clinvar):,} 个 ClinVar 突变（{int(clinvar.pathogenic.sum()):,} 个致病），所有预测器都有分的: {table.attrs['shared_variants']:,} 个")
table

### 结果怎么看
第一行字是一共多少个 ClinVar 突变、所有预测器都有分的有多少个。下面的表每个预测器一行，按"同一批突变"上的 AUROC 从高到低排。

**写这一章时我只用 300 个蛋白试过**：约 2,000 个 ClinVar 突变，所有预测器都有分的只有 130 个，数字很不稳，只看大概：

| 预测器 | AUROC（各自的突变） | AUROC（同一批 130 个） |
|---|---|---|
| AlphaMissense | 0.97 | 0.97 |
| 我们的功能分 | 0.92 | 0.87 |
| ESM-2 零样本 | 0.88 | 0.92 |
| 我们的稳定性分 | 0.83 | 0.80 |
| 旧 QAFI2 | 0.74 | 0.77 |

你跑完全部蛋白以后，数字会稳得多。看三件事：
1. **我们的功能分 vs ESM-2 零样本**：功能分 = 零样本 + 从 DMS 学到的东西。高出多少，就是我们学到的东西在临床突变上值多少；如果不高，也要如实写下来
2. **我们 vs AlphaMissense**：AlphaMissense 应该最高（理由见下面），关键是差距有多大
3. **稳定性分比功能分低**是正常的：很多致病突变不是通过破坏稳定性致病的（比如第 4 步 TP53 的 DNA 接触热点）

**要说清楚的几点**：
- AlphaMissense 训练时用了人群中的变异频率，和 ClinVar 的良性突变有重叠（常见变异多被判为良性），所以它在 ClinVar 上有一定的"主场优势"
- 我们的功能分来自 DMS 实验，**从没见过任何临床标注**，也从没见过这些蛋白。它在 ClinVar 上的成绩是一个干净的外部检验
- 旧 QAFI 论文报的 ClinVar AUROC（0.914）是在另一组精选的突变上，不能直接和这里比；这里在同一批突变上比，才公平

### 公司里什么情况怎么做
- **外部验证**是上线前最有说服力的一步：数据来源、标注方式都和训练数据无关
- 主动排除"见过"的部分（这里是我们数据里的全部蛋白），宁可数据少一点，也不能让成绩虚高

### 面试怎么说
> "As an external check I used ClinVar germline missense variants classified pathogenic or benign, restricted to proteins absent from all our training and test data, and compared AUROC on the variants every predictor covers."

---

## 第 8 步 · 新旧对比

### 在干什么
新一代的分数和旧 QAFI 的全蛋白组预测有多一致。

### 背景
比喻：**和上一代产品比**。不是比谁对，而是看换了这么多东西（特征、模型、标签）以后，结论变了多少。完全一样说明没进步；完全不同则要警惕。

随机抽 300 个蛋白，在每个蛋白内部算新功能分和旧 QAFI2、AlphaMissense 的相关系数（Pearson；AlphaMissense 是越高越致病，先取负号，方向和我们一致）：

In [ ]:
agreement = sql(w, f"""
    WITH proteins AS (SELECT uniprot_id FROM (SELECT DISTINCT uniprot_id FROM {CATALOG}.ml.proteome_scores) ORDER BY hash(uniprot_id) LIMIT 300)
    SELECT s.uniprot_id, corr(s.function_score, v.qafi2) AS pearson_with_qafi2, corr(s.function_score, -v.am_pathogenicity) AS pearson_with_alphamissense
    FROM {CATALOG}.ml.proteome_scores s
    JOIN {CATALOG}.silver.variant_scores v USING (uniprot_id, position, ref_aa, alt_aa)
    WHERE s.uniprot_id IN (SELECT uniprot_id FROM proteins)
    GROUP BY s.uniprot_id""").astype({"pearson_with_qafi2": float, "pearson_with_alphamissense": float})
agreement.describe().loc[["count", "25%", "50%", "75%"]].round(2)

### 结果怎么看
每个蛋白内部，新功能分和旧预测的相关系数（Pearson），看中位数（`50%`）：

写这一章时只有 300 个蛋白的分数，和旧预测重叠的约 30 个蛋白上：和旧 QAFI2 的中位数约 **0.71**，和 AlphaMissense 约 **0.69**。

- 0.6–0.8：大方向一致，又不完全一样。正常的换代
- 很高（> 0.9）：新模型和旧的几乎一样，换代没带来什么
- 很低（< 0.3）的蛋白：值得抽几个看看，是新的对还是旧的对

### 公司里什么情况怎么做
- 换代时一定做新旧对比，并且对**分歧最大**的那些蛋白抽查几个，弄明白是新的对还是旧的对
- 这个对比也是以后的"漂移"监控的起点：下一版模型再和这一版比

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 09c proteome-wide scoring, the product table and ClinVar validation") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 布局 | ESM 和模型在 Mac 上，数据在平台 | 流式推理：向量用完就扔 |
| 2 残基特征 | 同一个 SQL 宏，两张表 | 一段代码，训练和上线共用 |
| 3 一致性 | Python 和 SQL 在 84 万个突变上逐个比对 | 两份代码就要有一致性测试 |
| 4 试产 | 3 个蛋白，数量、分数、热点 | 先小批量，看熟悉的例子 |
| 5 全部 | 后台跑，可续跑，会让路 | 长任务要可续跑、进度可见 |
| 6 产品表 | 2.18 亿行，带版本；加一个视图 | 每一行都知道自己从哪来 |
| 7 ClinVar | 只用我们没见过的蛋白 | 外部验证最有说服力 |
| 8 新旧 | 和旧 QAFI 的一致性 | 换代要对比，分歧要抽查 |

**一句话**：项目的最终目标完成了，人类蛋白组的每一个可能错义突变都有了两个可追溯的分数，并且在临床数据上做了独立检验。

## 面试题

1. **How did you score 218 million variants without a GPU cluster?**
   > "Streaming inference on a laptop GPU: one ESM-2 pass per residue gives all 20 substitutions at once, embeddings are used and discarded, the job is chunked, resumable and memory-aware. About 5–6 hours."

2. **How do you prevent training-serving skew when features are reimplemented?**
   > "Share code where possible, here an SQL macro; where not, a parity test compares both implementations on the full training set before every release."

3. **What is your external validation and why is it credible?**
   > "ClinVar germline missense variants, pathogenic against benign, restricted to proteins absent from all our data, compared with published predictors on shared variants."

4. **What does your model not know?**
   > "Function that does not go through stability: TP53's DNA-contact hotspots score as tolerated. The training data is mostly stability assays, which is documented with the scores."

**下一步 10 · 对外服务**：把 `gold.fct_variant_effect` 导出到 AWS 的数据湖，用 Athena 查询，再用 API Gateway + Lambda 做一个按基因、按突变查分数的接口。